In [2]:

import re

def parse(expr):
    expr = expr.strip()
    match = re.fullmatch(r'([A-Za-z]\w*)\s*\((.*)\)', expr)

    if not match:
        return expr

    predicate = match.group(1)
    args = match.group(2).strip()

    if not args:
        return (predicate,)

    return (predicate,) + tuple(
        arg.strip() for arg in args.split(",")
    )


def is_variable(x):
    return isinstance(x, str) and x.islower() and len(x) == 1


def occurs(var, expr):
    if var == expr:
        return True
    if isinstance(expr, tuple):
        return any(occurs(var, arg) for arg in expr[1:])
    return False


def apply_subst(expr, subst):
    if isinstance(expr, str):
        while expr in subst:
            expr = subst[expr]
        return expr

    if isinstance(expr, tuple):
        return (expr[0],) + tuple(
            apply_subst(arg, subst) for arg in expr[1:]
        )

    return expr


def unify(x, y, subst):
    x = apply_subst(x, subst)
    y = apply_subst(y, subst)

    if x == y:
        return True

    if is_variable(x):
        if occurs(x, y):
            return False
        subst[x] = y
        return True

    if is_variable(y):
        if occurs(y, x):
            return False
        subst[y] = x
        return True

    if not isinstance(x, tuple) or not isinstance(y, tuple):
        return False

    if x[0] != y[0] or len(x) != len(y):
        return False

    for a, b in zip(x[1:], y[1:]):
        if not unify(a, b, subst):
            return False

    return True


# Main program
expr1 = input("Enter first expression: ")
expr2 = input("Enter second expression: ")

x = parse(expr1)
y = parse(expr2)
subst = {}

if unify(x, y, subst):
    print("Expressions can be unified.")
    print("Substitution set:", subst)
else:
    print("Expressions cannot be unified (FAILURE).")


Enter first expression: P(A,B,B)
Enter second expression: P(x,y,z)
Expressions can be unified.
Substitution set: {'x': 'A', 'y': 'B', 'z': 'B'}
